# 04 — Mercado de renda variável (Ibovespa)

Desenvolve a classe `RendaVariavel` (μ̂, Σ̂, amostragem). **F4.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
print('setup OK')

setup OK


## Desenvolvimento

A classe abaixo foi escrita aqui e, após os testes, movida para `app/mercado.py`.

In [ ]:
class RendaVariavel:
    """Mercado de renda variável (Ibovespa) — distribuição dos retornos. (F4)

    Opera no **espaço de retornos líquidos** (a mesma forma da tabela
    ``retornos``); o excesso ``R − R_f`` é formado depois, no agente/núcleo.
    """

    def __init__(self, retornos: pd.DataFrame, coluna_data: str = "data") -> None:
        """
        Parameters
        ----------
        retornos : DataFrame com (opcionalmente) a coluna ``data`` + uma coluna
            de retorno por ativo (decimal, sem NaN).
        coluna_data : nome da coluna de data a ignorar nos cálculos.
        """
        df = retornos.drop(columns=[coluna_data]) if coluna_data in retornos.columns else retornos.copy()
        self.ativos: list[str] = list(df.columns)
        self._R: np.ndarray = df.to_numpy(dtype=np.float64)  # (T, N)

        if self._R.ndim != 2 or self._R.shape[1] == 0:
            raise ValueError("retornos deve conter ao menos uma coluna de ativo.")
        if self._R.shape[0] < 2:
            raise ValueError("são necessárias ao menos 2 observações de retorno.")
        if np.isnan(self._R).any():
            raise ValueError("retornos não pode conter NaN.")

    @property
    def n_ativos(self) -> int:
        return self._R.shape[1]

    def media(self) -> np.ndarray:
        """Vetor de retornos esperados estimado μ̂, shape (N,). (F2, F4)"""
        return self._R.mean(axis=0)

    def covariancia(self) -> np.ndarray:
        """Matriz de covariância amostral não-viesada Σ̂, shape (N, N). (F2, F4)

        ``np.cov(R.T, ddof=1)`` + ``atleast_2d`` (idêntico ao estimador de
        referência ``estimate_sample_cov``).
        """
        return np.atleast_2d(np.cov(self._R.T, ddof=1))

    def amostrar(self, n: int, seed: int | None = None) -> np.ndarray:
        """Gera ``n`` cenários de retorno R ~ Normal(μ̂, Σ̂), shape (n, N).

        Usado pelo Monte Carlo da FOC (Etapa 1). Reprodutível via ``seed`` (NF4).

        Parameters
        ----------
        n : número de cenários.
        seed : semente do gerador (reprodutibilidade).
        """
        rng = np.random.default_rng(seed)
        mu = self.media()
        # R = μ + z·cholᵀ,  z ~ N(0, I)   ⇒  Cov(R) = Σ
        chol = np.linalg.cholesky(self.covariancia())
        z = rng.standard_normal((n, mu.shape[0]))
        return mu[None, :] + z @ chol.T


**Teste** — μ̂/Σ̂ vs numpy, `n_ativos` e `amostrar`.

In [3]:
rng = np.random.default_rng(0)
dados = pd.DataFrame({'data': pd.date_range('2000-01', periods=120, freq='MS').strftime('%Y-%m'),
                      'ibov': rng.normal(0.012, 0.05, 120), 'acao2': rng.normal(0.008, 0.04, 120)})
mv = RendaVariavel(dados); R = dados[['ibov','acao2']].to_numpy()
print('n_ativos:', mv.n_ativos, '| media:', mv.media())
assert np.allclose(mv.media(), R.mean(axis=0)) and np.allclose(mv.covariancia(), np.cov(R.T, ddof=1))
am = mv.amostrar(100_000, seed=1); print('amostra media ~ mu_hat:', am.mean(axis=0))
assert np.allclose(am.mean(axis=0), mv.media(), atol=1e-3)
print('F4 RendaVariavel: PASSOU')

n_ativos: 2 | media: [0.01606073 0.00295744]
amostra media ~ mu_hat: [0.01606869 0.00274092]
F4 RendaVariavel: PASSOU
